> Experiments with [xsa.access](../trapallada/xsa/access.py) module

In [3]:
import pathlib
import os
import tempfile


import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.plot

from astroquery.esa.xmm_newton import XMMNewton

So that the notebook is aware of modifications in the the `import`ed modules

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


All the available tables (only a few are printed)

In [5]:
tables = trapallada.xsa.access.get_all_tables()
tables[:5]

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


['public.dual',
 'tap_config.coord_sys',
 'tap_config.properties',
 'tap_schema.columns',
 'tap_schema.key_columns']

Get *metadata* for observations whose duration is in a given `duration_interval`

In [56]:
# obs_metadata_df = trapallada.xsa.access.get_observations_in_interval(duration_interval=(35_000, 45_000))
obs_metadata_df = trapallada.xsa.access.get_observations_in_interval(duration_interval=(35_000, 60_000))
obs_metadata_df

number of chunks: 2


,observation_id,duration
0,0006010301,35210
1,0006010401,35915
2,0006220201,46207
3,0008220201,42610
4,0008820401,54463
...,...,...
801,0851780101,52100
802,0936190101,43400
803,0943532801,59800
804,0943530601,36000


In order to locate a particular observation

In [54]:
# obs = '0200510101' # duration not in the requested interval!!
# obs = '0006010401'
obs = '0720750201'
obs_metadata_df[obs_metadata_df['observation_id'] == obs]

,observation_id,duration
1080,0720750201,44400


Javi's observation

In [55]:
javi_obs_id = '0605950101'
obs_metadata_df[obs_metadata_df['observation_id'] == javi_obs_id]

,observation_id,duration
780,0605950101,48315


In [6]:
# with tempfile.TemporaryDirectory(delete=True) as tmp_dir:
with tempfile.TemporaryDirectory(delete=False) as tmp_dir:

    # context's descriptor is turned into a `Pathlib`
    tmp_dir_root = pathlib.Path(tmp_dir)

    tar_file = trapallada.xsa.access.download_obs(obs, output_directory=tmp_dir_root)
    
    obs_df, header_df, mask_df = trapallada.xsa.util.read_tar(tar_file, max_n_sources=4)

INFO: https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0720750201&level=PPS&instname=PN [astroquery.esa.xmm_newton.core]
INFO: Wrote https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0720750201&level=PPS&instname=PN to /tmp/tmpriwhe6qa/0720750201.tar [astroquery.esa.xmm_newton.core]


Duration is

In [7]:
duration = obs_df.index[-1] - obs_df.index[0]
duration, duration.total_seconds()

(Timedelta('0 days 11:30:40.458362'), 41440.458362)

The directory where observations' data will be saved

In [8]:
output_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
output_directory.mkdir(exist_ok=True)

Let us select a few observations to download

In [9]:
obs_to_download = obs_metadata_df['observation_id'].iloc[:10]
obs_to_download

0    0006010301
1    0006010401
2    0008220201
3    0009650101
4    0009650201
5    0011420101
6    0011420201
7    0011430201
8    0011450201
9    0011830201
Name: observation_id, dtype: object

Downloading the first observation into a temporary directory

In [10]:
with tempfile.TemporaryDirectory(delete=False) as tmp_dir:

    # context's descriptor is turned into a `Pathlib`
    tmp_dir_root = pathlib.Path(tmp_dir)

    tar_file = trapallada.xsa.access.download_obs(obs_to_download[0], output_directory=tmp_dir_root)
print(tar_file)

INFO: https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0006010301&level=PPS&instname=PN [astroquery.esa.xmm_newton.core]
INFO: Wrote https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0006010301&level=PPS&instname=PN to /tmp/tmpujilkclo/0006010301.tar [astroquery.esa.xmm_newton.core]
/tmp/tmpujilkclo/0006010301.tar


In [11]:
# from astropy.table import Table
# tar_file
# trapallada.xsa.util.read_fits_file(tar_file)

# Low-level calls to `XMMNewton`

## `v_instrument_mode` table

*Instrument mode* table

In [12]:
instrument_mode_table = 'v_instrument_mode'
XMMNewton.get_columns('xsa.' + instrument_mode_table)

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


['instrument', 'instrument_mode_oid', 'mode', 'mode_friendly_name']

A sample

In [13]:
table = XMMNewton.query_xsa_tap(
    f'select TOP 20 * from {instrument_mode_table}',
)
table.to_pandas()

,instrument,instrument_mode_oid,mode,mode_friendly_name
0,EMOS1,27,CcdDiagnostic,Diagnostic 1x1
1,EMOS1,28,Diagnostic1x1,Diagnostic 1x1
2,EMOS1,29,Diagnostic3x3,Diagnostic 3x3
3,EMOS1,30,FastCompressed,Timing Compressed
4,EMOS1,31,FastUncompressed,Timing
5,EMOS1,32,OffsetVariance,Offset Var.
6,EMOS1,33,PrimeFullWindow,Full Frame
7,EMOS1,34,PrimePartialRFS,Refresh Frame Store
8,EMOS1,35,PrimePartialW2,Small Window
9,EMOS1,36,PrimePartialW3,Large Window


All the columns in the `v_all_observations`

In [14]:
XMMNewton.get_columns('xsa.v_all_observations') 

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


['bii',
 'citext',
 'dec',
 'duration',
 'end_utc',
 'footprint_fov',
 'heasarc_code',
 'lii',
 'moving_target',
 'observation_equatorial_spoint',
 'observation_fov_scircle',
 'observation_galactic_spoint',
 'observation_id',
 'observation_oid',
 'odf_proc_date',
 'odf_version',
 'position_angle',
 'pps_proc_date',
 'pps_version',
 'proposal_id',
 'proposal_oid',
 'proprietary_end_date',
 'ra',
 'ra_nom',
 'revolution',
 'sas_version',
 'seq_id',
 'start_utc',
 'stc_s',
 'with_science']